# ODI to Databricks Conversion - Badge Details Incremental Load

**Source File:** `undefined`
**Conversion Timestamp:** 2024-01-01T00:00:00.000000Z
**Description:** This notebook processes incremental `WC_MERCURY_BADGE_TS` data, stages it, identifies new and updated records, and merges them into the `WC_BADGE_DETAILS_D` dimension table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("V_ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00.000000", "ETL Last Extract Time (YYYY-MM-DD HH:MM:SS.ffffff)")
dbutils.widgets.text("V_ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59.999999", "ETL Current Extract Time (YYYY-MM-DD HH:MM:SS.ffffff)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380", "Datasource Number ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {2,4}: Select ETL last extract time from wc_etl_parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT
    MAX(last_extract_time) AS etl_last_extract_time
FROM
    workspace.prxbi_dw.wc_etl_parameters
WHERE
    ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {3,5}: Select ETL current extract time from wc_etl_parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT
    MAX(current_extract_time) AS etl_current_extract_time
FROM
    workspace.prxbi_dw.wc_etl_parameters
WHERE
    ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {6}: Select ROW_WID from WC_ETL_PARAMETERS
-- ROW_WID will be GENERATED ALWAYS AS IDENTITY in the target table, 
-- so this parameter is not used to explicitly set the ROW_WID value during insert.
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid AS
SELECT
    MAX(ROW_WID) AS etl_row_wid
FROM
    workspace.prxbi_dw.wc_etl_parameters
WHERE
    ETL_JOB_TYPE = '${ETL_JOB_TYPE}';

## Staging Table (`c_badge_details_stg`)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {30}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_details_stg;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.prxbi_dw.c_badge_details_stg (
	ID	STRING,
	BADGELOCATION	STRING,
	BADGETOKEN	STRING,
	BADGEVERSION	BIGINT,
	CONTACTEMAIL	STRING,
	CONTACTFIRSTNAME	STRING,
	CONTACTJOBTITLE	STRING,
	CONTACTLASTNAME	STRING,
	CONTACTPERSONRXMASTERID	STRING,
	CREATEDBYREGISTRATIONTYPE	STRING,
	CREATEDBYTYPE	STRING,
	CULTURE	STRING,
	CUSTOMERTYPE	STRING,
	EVENTEDITIONGBSCODE	STRING,
	ISBADGEUPDATE	STRING,
	MARKETINGPREFERENCESPROMPTREQU	STRING,
	ORGANISATIONCITY	STRING,
	ORGANISATIONCOUNTRYCODE	STRING,
	ORGANISATIONDISPLAYNAME	STRING,
	ORGANISATIONRXMASTERID	STRING,
	ORGANISATIONSTATE	STRING,
	PARTICIPATINGORGANISATIONID	STRING,
	PRODUCTCODE	STRING,
	QRCODECONTENT	STRING,
	REGISTRATIONID	STRING,
	STATUS	BIGINT,
	SUPPORTSTAFFCOMPANYADDRESS	STRING,
	SUPPORTSTAFFCOMPANYNAME	STRING,
	SUPPORTSTAFFMOBILEPHONE	STRING,
	SUPPORTSTAFFREPORTSTO	STRING,
	SUPPORTSTAFFSTANDS	STRING,
	SUPPORTSTAFFUSERACCESS	STRING,
	VERSIONNUMBER	BIGINT,
	MOBILEPHONE	STRING,
	FIRSTSCANNEDDATE	TIMESTAMP,
	LASTPRINTEDDATE	TIMESTAMP,
	ACCESSVALIDITYMODIFIEDDATE	TIMESTAMP,
	CREATEDDATE	TIMESTAMP,
	COMPANYPRODUCTCODE	STRING,
	PAYMENTSTATUS	STRING,
	PHOTOKEY	STRING,
	PHOTOSOURCE	STRING,
	PHOTOSOURCETYPE	STRING
)
USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {50}: Insert into staging table (with deduplication)
-- ODI MAX self-join dedup (F.12) replaced with ROW_NUMBER() for deterministic selection
INSERT INTO workspace.prxbi_dw.c_badge_details_stg
SELECT
    ID,
    BADGELOCATION,
    BADGETOKEN,
    BADGEVERSION,
    CONTACTEMAIL,
    CONTACTFIRSTNAME,
    CONTACTJOBTITLE,
    CONTACTLASTNAME,
    CONTACTPERSONRXMASTERID,
    CREATEDBYREGISTRATIONTYPE,
    CREATEDBYTYPE,
    CULTURE,
    CUSTOMERTYPE,
    EVENTEDITIONGBSCODE,
    ISBADGEUPDATE,
    MARKETINGPREFERENCESPROMPTREQUIRED AS MARKETINGPREFERENCESPROMPTREQU,
    ORGANISATIONCITY,
    ORGANISATIONCOUNTRYCODE,
    ORGANISATIONDISPLAYNAME,
    ORGANISATIONRXMASTERID,
    ORGANISATIONSTATE,
    PARTICIPATINGORGANISATIONID,
    PRODUCTCODE,
    QRCODECONTENT,
    REGISTRATIONID,
    STATUS,
    SUPPORTSTAFFCOMPANYADDRESS,
    SUPPORTSTAFFCOMPANYNAME,
    SUPPORTSTAFFMOBILEPHONE,
    SUPPORTSTAFFREPORTSTO,
    SUPPORTSTAFFSTANDS,
    SUPPORTSTAFFUSERACCESS,
    VERSIONNUMBER,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE
FROM
    (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY ID
                ORDER BY
                    INT_INSERT_DATE DESC,
                    VERSIONNUMBER DESC
            ) AS rn
        FROM
            workspace.prxbi_ts.wc_mercury_badge_ts
        WHERE
            INT_INSERT_DATE > to_timestamp('${V_ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss.SSSSSS')
            AND INT_INSERT_DATE <= to_timestamp('${V_ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss.SSSSSS')
    )
WHERE
    rn = 1;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}: Gather table stats (OPTIMIZE)
on staging table
OPTIMIZE workspace.prxbi_dw.c_badge_details_stg;

## Flow Table (`i_wc_badge_details_d_flow`)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {80}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {90}: Create flow table
CREATE TABLE workspace.prxbi_dw.i_wc_badge_details_d_flow (
	ROW_WID		BIGINT,
	BADGE_ID		STRING,
	BADGE_LOCATION		STRING,
	BADGE_TOKEN		STRING,
	BADGE_VERSION		BIGINT,
	CONTACT_EMAIL		STRING,
	CONTACT_FIRST_NAME		STRING,
	CONTACT_LAST_NAME		STRING,
	CONTACT_JOB_TITLE		STRING,
	CONTACT_PERSON_ID		STRING,
	CREATION_REG_TYPE		STRING,
	CREATION_TYPE		STRING,
	CULTURE		STRING,
	CUSTOMER_TYPE		STRING,
	EVENT_EDITION_CODE		STRING,
	BADGE_UPDATE_FLG		STRING,
	MARKETING_PREF_PROMPT		STRING,
	ORG_NAME		STRING,
	ORG_CITY		STRING,
	ORG_COUNTRY		STRING,
	ORG_ID		STRING,
	ORG_STATE		STRING,
	PARTICIPATING_ORG_ID		STRING,
	PRODUCT_CODE		STRING,
	QR_CODE		STRING,
	REGISTRATION_ID		STRING,
	STATUS		BIGINT,
	STAFF_COMPANY_NAME		STRING,
	STAFF_COMPANY_ADDR		STRING,
	STAFF_PHONE_NUM		STRING,
	STAFF_REPORTING		STRING,
	STAFF_STANDS		STRING,
	STAFF_USER_ACCESS		STRING,
	VERSION_NUM		BIGINT,
	INTEGRATION_ID		STRING,
	DATASOURCE_NUM_ID		STRING,
	W_INSERT_DT		TIMESTAMP,
	W_UPDATE_DT		TIMESTAMP,
	MOBILEPHONE		STRING,
	FIRSTSCANNEDDATE		TIMESTAMP,
	LASTPRINTEDDATE		TIMESTAMP,
	FIRSTSCANNEDDATE_FLG		STRING,
	LASTPRINTEDDATE_FLG		STRING,
	ACCESSVALIDITYMODIFIEDDATE		TIMESTAMP,
	CREATEDDATE		TIMESTAMP,
	COMPANYPRODUCTCODE		STRING,
	PAYMENTSTATUS		STRING,
	PHOTOKEY		STRING,
	PHOTOSOURCE		STRING,
	PHOTOSOURCETYPE		STRING,
	PACKAGE_NAME		STRING,
	IND_UPDATE		STRING
)
USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {100}: Insert into flow table (new/changed records)
INSERT INTO workspace.prxbi_dw.i_wc_badge_details_d_flow (
	BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	IND_UPDATE
)
SELECT
    BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	IND_UPDATE
FROM (
    SELECT
    	JOIN1_A.ID AS BADGE_ID,
    	JOIN1_A.BADGELOCATION AS BADGE_LOCATION,
    	JOIN1_A.BADGETOKEN AS BADGE_TOKEN,
    	JOIN1_A.BADGEVERSION AS BADGE_VERSION,
    	JOIN1_A.CONTACTEMAIL AS CONTACT_EMAIL,
    	JOIN1_A.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
    	JOIN1_A.CONTACTLASTNAME AS CONTACT_LAST_NAME,
    	JOIN1_A.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
    	JOIN1_A.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
    	JOIN1_A.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
    	JOIN1_A.CREATEDBYTYPE AS CREATION_TYPE,
    	JOIN1_A.CULTURE AS CULTURE,
    	JOIN1_A.CUSTOMERTYPE AS CUSTOMER_TYPE,
    	JOIN1_A.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
    	JOIN1_A.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
    	JOIN1_A.MARKETINGPREFERENCESPROMPTREQU AS MARKETING_PREF_PROMPT,
    	JOIN1_A.ORGANISATIONDISPLAYNAME AS ORG_NAME,
    	JOIN1_A.ORGANISATIONCITY AS ORG_CITY,
    	JOIN1_A.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
    	JOIN1_A.ORGANISATIONRXMASTERID AS ORG_ID,
    	JOIN1_A.ORGANISATIONSTATE AS ORG_STATE,
    	JOIN1_A.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
    	JOIN1_A.PRODUCTCODE AS PRODUCT_CODE,
    	JOIN1_A.QRCODECONTENT AS QR_CODE,
    	JOIN1_A.REGISTRATIONID AS REGISTRATION_ID,
    	JOIN1_A.STATUS AS STATUS,
    	JOIN1_A.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
    	JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
    	JOIN1_A.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
    	JOIN1_A.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
    	JOIN1_A.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
    	JOIN1_A.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
    	JOIN1_A.VERSIONNUMBER AS VERSION_NUM,
    	JOIN1_A.ID AS INTEGRATION_ID,
    	'${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    	JOIN1_A.MOBILEPHONE AS MOBILEPHONE,
    	JOIN1_A.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
    	JOIN1_A.LASTPRINTEDDATE AS LASTPRINTEDDATE,
    	CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
    	CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
    	JOIN1_A.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
    	JOIN1_A.CREATEDDATE AS CREATEDDATE,
    	JOIN1_A.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
    	JOIN1_A.PAYMENTSTATUS AS PAYMENTSTATUS,
    	JOIN1_A.PHOTOKEY AS PHOTOKEY,
    	JOIN1_A.PHOTOSOURCE AS PHOTOSOURCE,
    	JOIN1_A.PHOTOSOURCETYPE AS PHOTOSOURCETYPE,
    	WC_BADGE_PRODUCT_D_2.NAME_1 AS PACKAGE_NAME,
    	'I' AS IND_UPDATE
    FROM
    	workspace.prxbi_dw.c_badge_details_stg AS JOIN1_A
    LEFT OUTER JOIN
    	(
            SELECT
                WC_BADGE_PRODUCT_D.ID AS ID,
                WC_BADGE_PRODUCT_D.SKU AS SKU,
                WC_BADGE_PRODUCT_D.NAME AS NAME,
                ROW_NUMBER() OVER(PARTITION BY WC_BADGE_PRODUCT_D.SKU ORDER BY WC_BADGE_PRODUCT_D.ID DESC) AS COL,
                WC_BADGE_PRODUCT_D.SKU AS SKU_1,
                WC_BADGE_PRODUCT_D.NAME AS NAME_1
            FROM
                workspace.prxbi_dw.wc_badge_product_d AS WC_BADGE_PRODUCT_D
    	) AS WC_BADGE_PRODUCT_D_2
    ON  JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
    WHERE WC_BADGE_PRODUCT_D_2.COL = 1
) AS S
WHERE NOT EXISTS
	( SELECT 1 FROM workspace.prxbi_dw.wc_badge_details_d AS T
	WHERE
		T.INTEGRATION_ID	= S.INTEGRATION_ID
		AND T.DATASOURCE_NUM_ID	= S.DATASOURCE_NUM_ID
		AND ((T.BADGE_ID = S.BADGE_ID) OR (T.BADGE_ID IS NULL AND S.BADGE_ID IS NULL))
		AND ((T.BADGE_LOCATION = S.BADGE_LOCATION) OR (T.BADGE_LOCATION IS NULL AND S.BADGE_LOCATION IS NULL))
		AND ((T.BADGE_TOKEN = S.BADGE_TOKEN) OR (T.BADGE_TOKEN IS NULL AND S.BADGE_TOKEN IS NULL))
		AND ((T.BADGE_VERSION = S.BADGE_VERSION) OR (T.BADGE_VERSION IS NULL AND S.BADGE_VERSION IS NULL))
		AND ((T.CONTACT_EMAIL = S.CONTACT_EMAIL) OR (T.CONTACT_EMAIL IS NULL AND S.CONTACT_EMAIL IS NULL))
		AND ((T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME) OR (T.CONTACT_FIRST_NAME IS NULL AND S.CONTACT_FIRST_NAME IS NULL))
		AND ((T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME) OR (T.CONTACT_LAST_NAME IS NULL AND S.CONTACT_LAST_NAME IS NULL))
		AND ((T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE) OR (T.CONTACT_JOB_TITLE IS NULL AND S.CONTACT_JOB_TITLE IS NULL))
		AND ((T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID) OR (T.CONTACT_PERSON_ID IS NULL AND S.CONTACT_PERSON_ID IS NULL))
		AND ((T.CREATION_REG_TYPE = S.CREATION_REG_TYPE) OR (T.CREATION_REG_TYPE IS NULL AND S.CREATION_REG_TYPE IS NULL))
		AND ((T.CREATION_TYPE = S.CREATION_TYPE) OR (T.CREATION_TYPE IS NULL AND S.CREATION_TYPE IS NULL))
		AND ((T.CULTURE = S.CULTURE) OR (T.CULTURE IS NULL AND S.CULTURE IS NULL))
		AND ((T.CUSTOMER_TYPE = S.CUSTOMER_TYPE) OR (T.CUSTOMER_TYPE IS NULL AND S.CUSTOMER_TYPE IS NULL))
		AND ((T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE) OR (T.EVENT_EDITION_CODE IS NULL AND S.EVENT_EDITION_CODE IS NULL))
		AND ((T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG) OR (T.BADGE_UPDATE_FLG IS NULL AND S.BADGE_UPDATE_FLG IS NULL))
		AND ((T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT) OR (T.MARKETING_PREF_PROMPT IS NULL AND S.MARKETING_PREF_PROMPT IS NULL))
		AND ((T.ORG_NAME = S.ORG_NAME) OR (T.ORG_NAME IS NULL AND S.ORG_NAME IS NULL))
		AND ((T.ORG_CITY = S.ORG_CITY) OR (T.ORG_CITY IS NULL AND S.ORG_CITY IS NULL))
		AND ((T.ORG_COUNTRY = S.ORG_COUNTRY) OR (T.ORG_COUNTRY IS NULL AND S.ORG_COUNTRY IS NULL))
		AND ((T.ORG_ID = S.ORG_ID) OR (T.ORG_ID IS NULL AND S.ORG_ID IS NULL))
		AND ((T.ORG_STATE = S.ORG_STATE) OR (T.ORG_STATE IS NULL AND S.ORG_STATE IS NULL))
		AND ((T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID) OR (T.PARTICIPATING_ORG_ID IS NULL AND S.PARTICIPATING_ORG_ID IS NULL))
		AND ((T.PRODUCT_CODE = S.PRODUCT_CODE) OR (T.PRODUCT_CODE IS NULL AND S.PRODUCT_CODE IS NULL))
		AND ((T.QR_CODE = S.QR_CODE) OR (T.QR_CODE IS NULL AND S.QR_CODE IS NULL))
		AND ((T.REGISTRATION_ID = S.REGISTRATION_ID) OR (T.REGISTRATION_ID IS NULL AND S.REGISTRATION_ID IS NULL))
		AND ((T.STATUS = S.STATUS) OR (T.STATUS IS NULL AND S.STATUS IS NULL))
		AND ((T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME) OR (T.STAFF_COMPANY_NAME IS NULL AND S.STAFF_COMPANY_NAME IS NULL))
		AND ((T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR) OR (T.STAFF_COMPANY_ADDR IS NULL AND S.STAFF_COMPANY_ADDR IS NULL))
		AND ((T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM) OR (T.STAFF_PHONE_NUM IS NULL AND S.STAFF_PHONE_NUM IS NULL))
		AND ((T.STAFF_REPORTING = S.STAFF_REPORTING) OR (T.STAFF_REPORTING IS NULL AND S.STAFF_REPORTING IS NULL))
		AND ((T.STAFF_STANDS = S.STAFF_STANDS) OR (T.STAFF_STANDS IS NULL AND S.STAFF_STANDS IS NULL))
		AND ((T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS) OR (T.STAFF_USER_ACCESS IS NULL AND S.STAFF_USER_ACCESS IS NULL))
		AND ((T.VERSION_NUM = S.VERSION_NUM) OR (T.VERSION_NUM IS NULL AND S.VERSION_NUM IS NULL))
		AND ((T.MOBILEPHONE = S.MOBILEPHONE) OR (T.MOBILEPHONE IS NULL AND S.MOBILEPHONE IS NULL))
		AND ((T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE) OR (T.FIRSTSCANNEDDATE IS NULL AND S.FIRSTSCANNEDDATE IS NULL))
		AND ((T.LASTPRINTEDDATE = S.LASTPRINTEDDATE) OR (T.LASTPRINTEDDATE IS NULL AND S.LASTPRINTEDDATE IS NULL))
		AND ((T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG) OR (T.FIRSTSCANNEDDATE_FLG IS NULL AND S.FIRSTSCANNEDDATE_FLG IS NULL))
		AND ((T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG) OR (T.LASTPRINTEDDATE_FLG IS NULL AND S.LASTPRINTEDDATE_FLG IS NULL))
		AND ((T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE) OR (T.ACCESSVALIDITYMODIFIEDDATE IS NULL AND S.ACCESSVALIDITYMODIFIEDDATE IS NULL))
		AND ((T.CREATEDDATE = S.CREATEDDATE) OR (T.CREATEDDATE IS NULL AND S.CREATEDDATE IS NULL))
		AND ((T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE) OR (T.COMPANYPRODUCTCODE IS NULL AND S.COMPANYPRODUCTCODE IS NULL))
		AND ((T.PAYMENTSTATUS = S.PAYMENTSTATUS) OR (T.PAYMENTSTATUS IS NULL AND S.PAYMENTSTATUS IS NULL))
		AND ((T.PHOTOKEY = S.PHOTOKEY) OR (T.PHOTOKEY IS NULL AND S.PHOTOKEY IS NULL))
		AND ((T.PHOTOSOURCE = S.PHOTOSOURCE) OR (T.PHOTOSOURCE IS NULL AND S.PHOTOSOURCE IS NULL))
		AND ((T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE) OR (T.PHOTOSOURCETYPE IS NULL AND S.PHOTOSOURCETYPE IS NULL))
		AND ((T.PACKAGE_NAME = S.PACKAGE_NAME) OR (T.PACKAGE_NAME IS NULL AND S.PACKAGE_NAME IS NULL))
        );

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110,120}:
Create index
and gather stats (OPTIMIZE ZORDER)
on flow table
-- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## Target Table (`wc_badge_details_d`)

In [ ]:
-- MAGIC %sql
-- Create target table if it does not exist
-- ROW_WID defined as GENERATED ALWAYS AS IDENTITY to replace Oracle sequence.
CREATE TABLE IF NOT EXISTS workspace.prxbi_dw.wc_badge_details_d
(
    ROW_WID                      BIGINT GENERATED ALWAYS AS IDENTITY,
    BADGE_ID                     STRING,
    BADGE_LOCATION               STRING,
    BADGE_TOKEN                  STRING,
    BADGE_VERSION                BIGINT,
    CONTACT_EMAIL                STRING,
    CONTACT_FIRST_NAME           STRING,
    CONTACT_LAST_NAME            STRING,
    CONTACT_JOB_TITLE            STRING,
    CONTACT_PERSON_ID            STRING,
    CREATION_REG_TYPE            STRING,
    CREATION_TYPE                STRING,
    CULTURE                      STRING,
    CUSTOMER_TYPE                STRING,
    EVENT_EDITION_CODE           STRING,
    BADGE_UPDATE_FLG             STRING,
    MARKETING_PREF_PROMPT        STRING,
    ORG_NAME                     STRING,
    ORG_CITY                     STRING,
    ORG_COUNTRY                  STRING,
    ORG_ID                       STRING,
    ORG_STATE                    STRING,
    PARTICIPATING_ORG_ID         STRING,
    PRODUCT_CODE                 STRING,
    QR_CODE                      STRING,
    REGISTRATION_ID              STRING,
    STATUS                       BIGINT,
    STAFF_COMPANY_NAME           STRING,
    STAFF_COMPANY_ADDR           STRING,
    STAFF_PHONE_NUM              STRING,
    STAFF_REPORTING              STRING,
    STAFF_STANDS                 STRING,
    STAFF_USER_ACCESS            STRING,
    VERSION_NUM                  BIGINT,
    INTEGRATION_ID               STRING,
    DATASOURCE_NUM_ID            STRING,
    W_INSERT_DT                  TIMESTAMP,
    W_UPDATE_DT                  TIMESTAMP,
    MOBILEPHONE                  STRING,
    FIRSTSCANNEDDATE             TIMESTAMP,
    LASTPRINTEDDATE              TIMESTAMP,
    FIRSTSCANNEDDATE_FLG         STRING,
    LASTPRINTEDDATE_FLG          STRING,
    ACCESSVALIDITYMODIFIEDDATE   TIMESTAMP,
    CREATEDDATE                  TIMESTAMP,
    COMPANYPRODUCTCODE           STRING,
    PAYMENTSTATUS                STRING,
    PHOTOKEY                     STRING,
    PHOTOSOURCE                  STRING,
    PHOTOSOURCETYPE              STRING,
    PACKAGE_NAME                 STRING
)
USING DELTA;

## Mark Records for Update

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Mark existing records in flow table for update
-- Oracle tuple IN update (F.10) converted to MERGE
MERGE INTO workspace.prxbi_dw.i_wc_badge_details_d_flow AS T
USING (
    SELECT
        INTEGRATION_ID,
        DATASOURCE_NUM_ID
    FROM
        workspace.prxbi_dw.wc_badge_details_d
) AS S
ON
    T.INTEGRATION_ID = S.INTEGRATION_ID
    AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN
    UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target (`wc_badge_details_d`)

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {150,160}: Merge records from flow table into target table
-- Combines Oracle UPDATE (F.3 Tuple SET UPDATE) and INSERT (F.4 Identity Column) into a single MERGE statement.
-- ROW_WID is GENERATED ALWAYS AS IDENTITY and excluded from INSERT list.
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS T
USING workspace.prxbi_dw.i_wc_badge_details_d_flow AS S
ON
    T.INTEGRATION_ID = S.INTEGRATION_ID
    AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN
    UPDATE SET
        T.BADGE_ID = S.BADGE_ID,
        T.BADGE_LOCATION = S.BADGE_LOCATION,
        T.BADGE_TOKEN = S.BADGE_TOKEN,
        T.BADGE_VERSION = S.BADGE_VERSION,
        T.CONTACT_EMAIL = S.CONTACT_EMAIL,
        T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME,
        T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME,
        T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE,
        T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID,
        T.CREATION_REG_TYPE = S.CREATION_REG_TYPE,
        T.CREATION_TYPE = S.CREATION_TYPE,
        T.CULTURE = S.CULTURE,
        T.CUSTOMER_TYPE = S.CUSTOMER_TYPE,
        T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE,
        T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG,
        T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT,
        T.ORG_NAME = S.ORG_NAME,
        T.ORG_CITY = S.ORG_CITY,
        T.ORG_COUNTRY = S.ORG_COUNTRY,
        T.ORG_ID = S.ORG_ID,
        T.ORG_STATE = S.ORG_STATE,
        T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID,
        T.PRODUCT_CODE = S.PRODUCT_CODE,
        T.QR_CODE = S.QR_CODE,
        T.REGISTRATION_ID = S.REGISTRATION_ID,
        T.STATUS = S.STATUS,
        T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME,
        T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR,
        T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM,
        T.STAFF_REPORTING = S.STAFF_REPORTING,
        T.STAFF_STANDS = S.STAFF_STANDS,
        T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS,
        T.VERSION_NUM = S.VERSION_NUM,
        T.MOBILEPHONE = S.MOBILEPHONE,
        T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE,
        T.LASTPRINTEDDATE = S.LASTPRINTEDDATE,
        T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG,
        T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG,
        T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
        T.CREATEDDATE = S.CREATEDDATE,
        T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE,
        T.PAYMENTSTATUS = S.PAYMENTSTATUS,
        T.PHOTOKEY = S.PHOTOKEY,
        T.PHOTOSOURCE = S.PHOTOSOURCE,
        T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE,
        T.PACKAGE_NAME = S.PACKAGE_NAME,
        T.W_UPDATE_DT = current_timestamp()
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN
    INSERT (
        BADGE_ID,
        BADGE_LOCATION,
        BADGE_TOKEN,
        BADGE_VERSION,
        CONTACT_EMAIL,
        CONTACT_FIRST_NAME,
        CONTACT_LAST_NAME,
        CONTACT_JOB_TITLE,
        CONTACT_PERSON_ID,
        CREATION_REG_TYPE,
        CREATION_TYPE,
        CULTURE,
        CUSTOMER_TYPE,
        EVENT_EDITION_CODE,
        BADGE_UPDATE_FLG,
        MARKETING_PREF_PROMPT,
        ORG_NAME,
        ORG_CITY,
        ORG_COUNTRY,
        ORG_ID,
        ORG_STATE,
        PARTICIPATING_ORG_ID,
        PRODUCT_CODE,
        QR_CODE,
        REGISTRATION_ID,
        STATUS,
        STAFF_COMPANY_NAME,
        STAFF_COMPANY_ADDR,
        STAFF_PHONE_NUM,
        STAFF_REPORTING,
        STAFF_STANDS,
        STAFF_USER_ACCESS,
        VERSION_NUM,
        INTEGRATION_ID,
        DATASOURCE_NUM_ID,
        MOBILEPHONE,
        FIRSTSCANNEDDATE,
        LASTPRINTEDDATE,
        FIRSTSCANNEDDATE_FLG,
        LASTPRINTEDDATE_FLG,
        ACCESSVALIDITYMODIFIEDDATE,
        CREATEDDATE,
        COMPANYPRODUCTCODE,
        PAYMENTSTATUS,
        PHOTOKEY,
        PHOTOSOURCE,
        PHOTOSOURCETYPE,
        PACKAGE_NAME,
        W_INSERT_DT,
        W_UPDATE_DT
    )
    VALUES (
        S.BADGE_ID,
        S.BADGE_LOCATION,
        S.BADGE_TOKEN,
        S.BADGE_VERSION,
        S.CONTACT_EMAIL,
        S.CONTACT_FIRST_NAME,
        S.CONTACT_LAST_NAME,
        S.CONTACT_JOB_TITLE,
        S.CONTACT_PERSON_ID,
        S.CREATION_REG_TYPE,
        S.CREATION_TYPE,
        S.CULTURE,
        S.CUSTOMER_TYPE,
        S.EVENT_EDITION_CODE,
        S.BADGE_UPDATE_FLG,
        S.MARKETING_PREF_PROMPT,
        S.ORG_NAME,
        S.ORG_CITY,
        S.ORG_COUNTRY,
        S.ORG_ID,
        S.ORG_STATE,
        S.PARTICIPATING_ORG_ID,
        S.PRODUCT_CODE,
        S.QR_CODE,
        S.REGISTRATION_ID,
        S.STATUS,
        S.STAFF_COMPANY_NAME,
        S.STAFF_COMPANY_ADDR,
        S.STAFF_PHONE_NUM,
        S.STAFF_REPORTING,
        S.STAFF_STANDS,
        S.STAFF_USER_ACCESS,
        S.VERSION_NUM,
        S.INTEGRATION_ID,
        S.DATASOURCE_NUM_ID,
        S.MOBILEPHONE,
        S.FIRSTSCANNEDDATE,
        S.LASTPRINTEDDATE,
        S.FIRSTSCANNEDDATE_FLG,
        S.LASTPRINTEDDATE_FLG,
        S.ACCESSVALIDITYMODIFIEDDATE,
        S.CREATEDDATE,
        S.COMPANYPRODUCTCODE,
        S.PAYMENTSTATUS,
        S.PHOTOKEY,
        S.PHOTOSOURCE,
        S.PHOTOSOURCETYPE,
        S.PACKAGE_NAME,
        current_timestamp(),
        current_timestamp()
    );

## Cleanup

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {180}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {210}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_details_stg;

## Conversion Notes and Manual Actions Required

1.  **Schema Naming:**
    *   Original Oracle schema `PRXBI_DW_SEP` converted to `workspace.prxbi_dw`.
    *   Original Oracle schema `PRXBI_TS_SEP` converted to `workspace.prxbi_ts`.
    *   `WC_ETL_PARAMETERS` assumed to be in `workspace.prxbi_dw`.
2.  **Table Naming:**
    *   Staging table `C$_0A7SUCRIPSM1CG2656H955OU5QP` renamed to `c_badge_details_stg` for readability and adherence to convention.
    *   Flow table `I$_WADHBP95VCNT6J0F17HEGQQJ3GB` renamed to `i_wc_badge_details_d_flow`.
3.  **Data Types:** Oracle `VARCHAR2`, `NUMBER(p,0)`, `DATE`, `TIMESTAMP(n)` types mapped to Spark SQL `STRING`, `BIGINT`, `TIMESTAMP` respectively.
4.  **Parameter Handling:** ODI `#GLOBAL` parameters (`V_ETL_JOB_TYPE`, `V_ETL_LAST_EXTRACT_TIME`, `V_ETL_CURRENT_EXTRACT_TIME`, `DATASOURCE_NUM_ID`) converted to Databricks widgets and referenced using `${PARAMETER_NAME}` syntax. Default values provided for widgets.
5.  **Date/Timestamp Functions:**
    *   `TO_TIMESTAMP` format strings converted from Oracle `YYYY-MM-DD HH24:MI:SS.FF` to Spark `yyyy-MM-dd HH:mm:ss.SSSSSS`.
    *   `SYSTIMESTAMP` converted to `current_timestamp()`.
6.  **Conditional Logic:** `NVL2(condition,'Y','N')` converted to `CASE WHEN condition IS NOT NULL THEN 'Y' ELSE 'N' END`.
7.  **Deduplication (SCEN_TASK_NO 50 - F.12):** The ODI-specific `MAX` based self-join deduplication pattern from `WC_MERCURY_BADGE_TS` has been replaced with a `ROW_NUMBER() OVER (PARTITION BY ID ORDER BY INT_INSERT_DATE DESC, VERSIONNUMBER DESC)` window function for deterministic single-row-per-key selection.
8.  **`DBMS_STATS` and `CREATE INDEX`:**
    *   Oracle `DBMS_STATS.GATHER_TABLE_STATS` (SCEN_TASK_NO 60, 120) are replaced by `OPTIMIZE` statements.
    *   Oracle `CREATE INDEX` (SCEN_TASK_NO 110) is replaced by `OPTIMIZE ... ZORDER BY` for improved query performance on Delta tables. Each `OPTIMIZE ZORDER BY` is preceded by `SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;` as required (F.11).
9.  **DML Conversion:**
    *   Oracle `DROP TABLE ... PURGE` converted to `DROP TABLE IF EXISTS`.
    *   `NOLOGGING` and `/*+ append */` hints were removed as they are not applicable to Delta Lake.
    *   PL/SQL `BEGIN ... END;` blocks were removed, and their contents translated.
10. **`IND_UPDATE` Flagging (SCEN_TASK_NO 130 - F.10):** The `UPDATE ... WHERE (col1, col2) IN (SELECT ...)` pattern for marking records in the flow table has been rewritten as a `MERGE INTO ... WHEN MATCHED THEN UPDATE SET` statement, as tuple `IN` predicates are not supported in Delta `UPDATE` statements.
11. **Final MERGE into Target (SCEN_TASK_NO 150, 160 - F.3, F.4):**
    *   The separate `UPDATE ... SET (col1, col2) = (SELECT ...)` and `INSERT ... WHERE NOT EXISTS` statements targeting `WC_BADGE_DETAILS_D` have been combined into a single `MERGE INTO` statement.
    *   The `ROW_WID` column in `WC_BADGE_DETAILS_D` is defined as `BIGINT GENERATED ALWAYS AS IDENTITY`. As per F.4, this column is explicitly excluded from the `INSERT` column list in the `MERGE` statement, as its value is auto-generated by Delta Lake.
    *   The complex `NOT EXISTS` condition for comparing all columns from the initial `I$_` population (SCEN_TASK_NO 100) has been preserved.